https://www.howtoforge.com/tutorial/how-to-use-ftp-on-the-linux-shell/

Order files here
Use ftp via ftp ftp2.archive.arm.gov
Login with User and Password
Name: franzihe
Password:

Change directory cd /hellmuthf2/232434

Download files with FTP by setting local FTP file download directory 
lcd /scratch/franzihe/input/COMBLE/mwr

mget *anxmwr*

In [1]:
# rsync -av --progress /scratch/franzihe/input/COMBLE u6030275@notchpeak.chpc.utah.edu:/uufs/chpc.utah.edu/common/home/coopers-group1/Franziska/data 

In [2]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]


if "mimi" in hostname:
    DATA_DIR = "/scratch/franzihe/"
    FIG_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/Figures/COMBLE/"

elif "augustiner" in hostname: 
    DATA_DIR = "/home/franzihe/Data/"
    FIG_DIR = "/home/franzihe/Documents/Figures/COMBLE/"

elif "wessel" in hostname:
    DATA_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/"
    FIG_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/Figures/COMBLE/"


INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# make figure directory to put in reflectivity processed data
try:
    new_dir = pathlib.Path(FIG_DIR)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

mimi.uio.no


## Import python packages

* `Python` environment requirements: file requirements_retrieval.txt
* load `python` packages from imports.py
* load `functions` from functions.py

In [3]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(xr, glob, plt,radar, np, DateFormatter, datetime, cm, pd, da, fct)


In [4]:
# reload imports
%load_ext autoreload
%autoreload 2

In [18]:
sites = ["COMBLE"]
years = ["2019","2020"]
months = ["12", "01", "02", "03", "04", "05"]
days = np.arange(1, 32)

title = ['MRR reflectivity', 'MRR mean Doppler Velocity', 'Precipitation accumulation', '2-m Air temperature', '10-m Wind']

attrs = {'title': 'COMBLE ARM Products: KAZR, armbeatm, interpolatedsonde '
 'Conventions': 'ARM-1.2, ARM-1.2, ARM-1.1', 
 'process_version': 'ingest-kazrcfr-1.2-0.el7, vap-armbeatm-1.0-0.dev6.dirty.el7, vap-interpolatedsonde-6.6-0.el6',
 'dod_version':', ,',
 'site_id': ', ',
 'platform_id': ', ',
 'facility_id': ', ,',
 'data_level': ', ,',
 'location_description': ', ,',
 'institution': 'Department of Energy Atmospheric Radiation Measurement Program',
 'doi': '10.5439/1478370, ,',
 'history': 'created at 2020-06-04 13:18:21'
 'Municipality': 'Andøy', 
 'County': 'Nordland',
 'Creator': 'Franziska Hellmuth',
 'Institution': 'Dept. of Geoscience, University of Oslo'
 }

In [ ]:
for site in sites:
    for year in years:
        armbeatm_in = sorted(glob(os.path.join(INPUT_DATA_DIR, '{}/armbeatm*/*{}*.nc'.format(site, year))))
        ds_armbeatm = xr.open_mfdataset(armbeatm_in, decode_times=True)
        ds_armbeatm.coords['time'] = fct.adjust_time(ds_armbeatm)

        for month in months:
            for day in days:
                if day < 10:
                    day = "0{}".format(day)

                date = "{}{}{}".format(year, month, day)
            
                # Define name for figure
                figname = 'kazr_reflectivity_{}_{}.png'.format(site, date)



                ## load daily files
                # KAZR
                kazr_in = sorted(glob(os.path.join(INPUT_DATA_DIR, '{}/kazr*/*{}*.nc'.format(site,date))))
                # Sonde
                intpsonde_in = sorted(glob(os.path.join(INPUT_DATA_DIR, '{}/interpolatedsonde*/*{}*.nc'.format(site,date))))
                if len(kazr_in) == 0 or len(intpsonde_in) == 0:
                    continue
                else:
                                
                    # Define start and end time of individual day
                    day_start = datetime.datetime.strptime("{} 00:00:00.00".format(date), '%Y%m%d %H:%M:%S.%f').strftime('%Y-%m-%dT%H:%M:%S.%f')
                    day_end = (datetime.datetime.strptime("{} 00:00:00.00".format(date), '%Y%m%d %H:%M:%S.%f') + datetime.timedelta(hours=23, minutes=59)).strftime('%Y-%m-%dT%H:%M:%S.%f')
                    _xmax = (datetime.datetime.strptime(day_end, '%Y-%m-%dT%H:%M:%S.%f') + datetime.timedelta(minutes=1,)).strftime('%Y-%m-%dT%H:%M:%S.%f')
                    
                    # KAZR
                    ds_dict = {}
                    for hours in range(len(kazr_in)):
                        ds_dict[hours] = xr.open_dataset(kazr_in[hours], decode_times=True)

                        ds_dict[hours].coords['time'] = fct.adjust_time(ds_dict[hours])

                    # create dataset with all hours    
                    _ds = list(ds_dict.values())
                    _coord = list(ds_dict.keys())
                    ds_kazr = xr.concat(objs=_ds, dim='time', coords="all")
                    
                    # Sonde
                    ds_intpsonde = xr.open_mfdataset(intpsonde_in, decode_times=True)
                    ds_intpsonde.coords['time'] = fct.adjust_time(ds_intpsonde)

                    # set height to m
                    _attrs = ds_intpsonde.height.attrs
                    _attrs['units']  ='m'
                    _attrs['comment'] = 'Height represents m above mean sea level'
                    ds_intpsonde = ds_intpsonde.assign_coords({"height":('height',(ds_intpsonde.height*1000).values,ds_intpsonde.height.attrs)})

                    # load variables
                    # select reflectivity to make radar reflectivity plot
                    _Ze = ds_kazr['reflectivity'].sel(time=slice(day_start, day_end))
                    _Ze = _Ze.resample(time='60S').mean(dim='time', skipna=True, keep_attrs=True) # minutely average

                    # select mean_doppler_velocity
                    _W = ds_kazr['mean_doppler_velocity'].sel(time = slice(day_start, day_end))
                    _W = _W.resample(time='60S').mean(dim='time', skipna=True, keep_attrs=True) # minutely average

                    _pp = ds_armbeatm.precip_rate_sfc.sel(time = slice(day_start, day_end))
                    _air_temp = (ds_armbeatm.temperature_sfc - 273.15).sel(time = slice(day_start, day_end))
                    _air_temp = _air_temp.assign_attrs(ds_armbeatm.temperature_sfc.attrs).assign_attrs({'units':'degC'})

                    # _air_temp_h = (ds_armbeatm.temperature_h - 273.15).sel(time = slice(day_start, day_end))
                    # _air_temp_h = _air_temp_h.assign_attrs(ds_armbeatm.temperature_h.attrs).assign_attrs({'units':'degC'})
                    _u_sfc = ds_armbeatm.u_wind_sfc.sel(time = slice(day_start, day_end))
                    _v_sfc = ds_armbeatm.v_wind_sfc.sel(time = slice(day_start, day_end))
                    # _u_h = ds_armbeatm.u_wind_h
                    # _v_h = ds_armbeatm.v_wind_h
                    _air_temp_h = ds_intpsonde.temp.sel(time=slice(day_start,day_end))

                    ### Make figure
                    fig, axsm = plt.subplots(5, 1, figsize=[10,12], sharex=True, )
                    axs = axsm.flatten()

                    # Reflectivity and interpolated temperature
                    contf = _Ze.plot.contourf(ax = axs[0], x='time', y='range', vmin = -20, 
                                                levels = np.arange(-20, 30,0.5), 
                                                cmap=plt.get_cmap('pymeteo_radar'), extend='max', 
                                                add_colorbar=False,
                                                )
                    cbaxes = fig.add_axes([0.92, 0.8, 0.025, 0.14])
                    cbar = plt.colorbar(contf, cax=cbaxes, shrink=0.5, orientation='vertical')
                    cbar.ax.set_ylabel('Ze ({})'.format(_Ze.attrs['units']))

                    cont1 = _air_temp_h.plot.contour(ax = axs[0], x='time', levels = np.arange(-65,20,5), cmap=cm.devon_r, add_colorbar =False)
                    cbaxes = fig.add_axes([1.01, 0.8, 0.025, 0.14])
                    cbar = plt.colorbar(cont1, cax=cbaxes, shrink=0.5, orientation='vertical')
                    cbar.ax.set_ylabel('Temperature ({})'.format(_air_temp_h.attrs['units']))


                    # vertical Doppler Velocity
                    p2 = _W.plot.contourf(ax = axs[1], x='time', y='range',vmin=-3, vmax=3, 
                                            levels=np.arange(-3,3.5,0.5), extend='both', cmap=cm.vik, add_colorbar=False,)
                    cbaxes = fig.add_axes([0.92, 0.625, 0.025, 0.14])
                    cbar = plt.colorbar(p2, cax=cbaxes, shrink=0.5, orientation='vertical')
                    cbar.ax.set_ylabel('W ({})'.format(_W.attrs['units']))

                    # Precipitation accumulation
                    axs[2].bar(_pp['time'].values, _pp.values, width=0.025, align='center')
                    axs[2].set_ylabel('Precipitation (mm h-1)')
                    axs[2].set_ylim([0, 4])  



                    # 2-m temperature
                    _air_temp.plot(ax = axs[3], x='time')
                    axs[3].set_ylabel('2-m temperature ({})'.format(_air_temp.attrs['units']))
                    axs[3].hlines(y= 2., xmin=day_start, xmax=_xmax, color = 'grey')
                    axs[3].set_ylim([-11.5, 12.5])

                    # Define the date format
                    date_form = DateFormatter("%H:00")
                    for ax, tt in zip(axs,title):
                                ax.set( xlabel = "",
                                        title =tt
                                        )
                                ax.xaxis.grid()
                                # ax.set_xlim([day_start, _xmax])
                                ax.tick_params(axis='x', labelrotation=0, )
                                plt.setp(ax.get_xticklabels(), horizontalalignment="center")
                                ax.xaxis.set_major_formatter(date_form)

                    for ax in axs[0:2]:
                        ax.set(ylabel = 'Height (m)',)
                        ax.set_ylim([0, 10000])

                    axs[-1].set(xlabel="Time (UTC)")

                    fig.suptitle("{} ({})".format(site, _Ze.time.dt.date[0].values), fontweight="bold")
                    fig.subplots_adjust(top=0.94)


                    # save image to png
                    plt.savefig(FIG_DIR + figname, format='png', bbox_inches='tight', transparent=True)

                    # create dataset with all varibles
                    _Ze_mean = xr.DataArray(data=da.full(shape = (len(_Ze.time),len(range(200, (int(_Ze.range[-1].values)+100),200))),
                                                        fill_value=np.nan), 
                                            coords={'time':_Ze.time.values, 'range':range(200, (int(_Ze.range[-1].values)+100),200)})
                    _W_mean = xr.DataArray(data=da.full(shape = (len(_W.time),len(range(200, (int(_W.range[-1].values)+100),200))),
                                                        fill_value=np.nan), 
                                            coords={'time':_W.time.values, 'range':range(200, (int(_W.range[-1].values)+100),200)})
                    
                    for i, k in zip(range(100, (int(_Ze.range[-1].values)+200),200), range(len(range(200, (int(_Ze.range[-1].values)+100),200)))):
                        _Z = 10**(_Ze.sel(range= slice(i, i+200))/10)
                        _Z_mean = _Z.mean('range', skipna=True, keep_attrs=True)
                        _Ze_mean[:,k] = 10*np.log10(_Z_mean)

                        _W_mean[:,k] = _W.sel(range= slice(i, i+200)).mean('range', skipna=True, keep_attrs=True)

                    _Ze_mean = _Ze_mean.assign_attrs(_Ze.attrs)
                    _W_mean = _W_mean.assign_attrs(_W.attrs)

                    # create dataset with all variables
                    dso = xr.Dataset(attrs=attrs)

                    dso['Ze'] = _Ze_mean
                    dso['W'] = _W_mean
                    dso['2m_temperature'] = _air_temp
                    dso['air_temperature'] = _air_temp_h
                    dso['precipitation'] = _pp
                    # dso['wind_direction'] = _wd
                    # dso['wind_speed'] = _ws

                    file_out = "{}_{}.nc".format(os.path.join(OUTPUT_DATA_DIR, 'COMBLE/bin_averaged/'), site, date)
                    dso.to_netcdf(file_out)

In [57]:
ds_kazr.attrs['process_version']

'ingest-kazrcfr-1.2-0.el7'

In [55]:
, ds_armbeatm.attrs['process_version'], ds_intpsonde.attrs['process_version']

('ingest-kazrcfr-1.2-0.el7',
 'vap-armbeatm-1.0-0.dev6.dirty.el7',
 'vap-interpolatedsonde-6.6-0.el6')